# Почему падает сенсор `check_new_data`

Сенсор запуска с Logical date D (сам запуск D+1 в 12:00) ждёт по каждому потоку ODS успешный запуск со стартом не раньше D+1 00:00. На сервере он сейчас сдаётся в 16:00.

1. Журнал `ekm.tm_task_run` по 17 потокам с 30.10.
2. Таблица: поток × день, время первого успешного старта.
3. Что увидел бы сенсор в каждый день: каких потоков не хватало к 16:00 и к 11:00 следующего дня.
4. Нет ли потоков под другим именем.

Только чтение. Логин keytab спрашивается один раз.

In [ ]:
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)

SCHEDULE_TABLE = "ekm.tm_task_run"
SINCE = "2026-10-30"
SENSOR_GIVE_UP_HOUR = 16

RUN_NAMES = [
    "APP_ODS/wf_ods2dl_ora2hive_scd1_agreements",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_companies",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_agr_terms",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_trx",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_trx_acq",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_trx_int",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_base24_fiids",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_merchants",
    "APP_ODS/wf_ods2dl_ora2hive_scd1_pos_terminals",
    "APP_ODS_EFR/wf_ods2dl_ora2hive_scd1_mrc_pos_rent",
    "App_ODS_CFT_D4/wf_ods2dl_ora2hive_scd1_z_r2_ip_merchants",
    "App_ODS_CFT_D4/wf_ods2dl_ora2hive_scd1_z_r2_ip_dog_oper",
    "App_ODS_CFT_D4/wf_ods2dl_ora2hive_scd1_z_r2_tariff_plan",
    "App_ODS_CFT_D1/wf_ods2dl_ora2hive_scd1_z_client",
    "App_ODS_CFT_D1/wf_ods2dl_ora2hive_scd1_z_branch",
    "App_ODS_CFT_D1/wf_ods2dl_ora2hive_scd1_z_cl_corp",
    "App_ODS_CFT_D2/wf_ods2dl_ora2hive_scd1_z_depart",
]
names = [n.lower() for n in RUN_NAMES]

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def imp_fetch(sql):
    with imp:
        return imp.fetch(sql)


with imp:
    imp.execute(f"REFRESH {SCHEDULE_TABLE}")
print("OK")

## 1. Журнал запусков 17 потоков

In [ ]:
in_list = ", ".join(f"'{n}'" for n in names)
log = imp_fetch(f"""
    SELECT lower(run_name) AS run_name,
           cast(run_start_dt AS timestamp) AS run_start_dt,
           cast(run_status AS string) AS run_status
    FROM {SCHEDULE_TABLE}
    WHERE cast(run_start_dt AS timestamp) >= cast('{SINCE}' AS timestamp)
      AND lower(run_name) IN ({in_list})
""")
log = log if log is not None else pd.DataFrame(columns=["run_name", "run_start_dt", "run_status"])
log["run_start_dt"] = pd.to_datetime(log["run_start_dt"], errors="coerce")
log["day"] = log["run_start_dt"].dt.date
log["flow"] = log["run_name"].str.split("/").str[-1].str.replace("wf_ods2dl_ora2hive_scd1_", "", regex=False)

print("Строк в журнале:", len(log))
print("Статусы:")
display(log["run_status"].value_counts().to_frame("runs"))

absent = sorted(set(names) - set(log["run_name"]))
print("Потоков нет в журнале совсем с", SINCE, ":", absent or "нет, все есть")

## 2. Поток × день: время первого успешного старта

Пусто — в этот день успешного запуска не было. Время 23:xx — поток стартует до полуночи, и сенсор его не видит.

In [ ]:
ok = log[log["run_status"] == "Succeeded"]
first_ok = ok.groupby(["flow", "day"])["run_start_dt"].min().dt.strftime("%H:%M")
pivot = first_ok.unstack("day").reindex(
    [n.split("/")[-1].replace("wf_ods2dl_ora2hive_scd1_", "") for n in names]
)
display(pivot.fillna(""))

print("Обычное время успешного старта по потокам (часы):")
display(
    ok.assign(hour=ok["run_start_dt"].dt.hour)
    .groupby("flow")["hour"].agg(["min", "median", "max", "count"])
    .sort_values("median")
)

## 3. Что видел бы сенсор

`load_day` — день самого запуска DAG (Logical date + 1). Для каждого дня: каких потоков не хватало к 16:00 (как сейчас на сервере) и к 11:00 следующего дня (новая версия).

In [ ]:
rows = []
for load_day in pd.date_range(SINCE, pd.Timestamp.now().normalize()):
    for label, deadline in [
        ("к 16:00", load_day + pd.Timedelta(hours=SENSOR_GIVE_UP_HOUR)),
        ("к 11:00 след. дня", load_day + pd.Timedelta(hours=35)),
    ]:
        seen = set(ok[(ok["run_start_dt"] >= load_day) & (ok["run_start_dt"] < deadline)]["run_name"])
        missing = [n.split("/")[-1].replace("wf_ods2dl_ora2hive_scd1_", "") for n in names if n not in seen]
        rows.append({
            "load_day": load_day.date(),
            "weekday": load_day.strftime("%a"),
            "deadline": label,
            "missing_cnt": len(missing),
            "missing": ", ".join(missing),
        })
display(pd.DataFrame(rows))

## 4. Нет ли этих потоков под другим именем

Ищем в журнале любые имена с той же таблицей (`scd1_agreements` и т.д.) за последние 3 дня.

In [ ]:
suffixes = sorted({n.split("wf_ods2dl_ora2hive_")[-1] for n in names})
like = " OR ".join(f"lower(run_name) LIKE '%{s}%'" for s in suffixes)
since_3d = (pd.Timestamp.now().normalize() - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
alt = imp_fetch(f"""
    SELECT lower(run_name) AS run_name,
           cast(run_status AS string) AS run_status,
           count(*) AS runs,
           min(cast(run_start_dt AS timestamp)) AS first_start,
           max(cast(run_start_dt AS timestamp)) AS last_start
    FROM {SCHEDULE_TABLE}
    WHERE cast(run_start_dt AS timestamp) >= cast('{since_3d}' AS timestamp)
      AND ({like})
    GROUP BY 1, 2
    ORDER BY 1, 2
""")
if alt is not None and len(alt):
    alt["in_sensor"] = alt["run_name"].isin(names)
display(alt)